In [1]:
import matplotlib.pyplot as plt
import scanpy as sc
import numpy as np
import os

import bin2cell as b2c

#create directory for stardist input/output files
os.makedirs("stardist", exist_ok=True)

bioimageio_utils.py (2): pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


# Read in data

In [2]:
dir_path = "/rfs/project/rfs-iCNyzSAaucw/jc2226/heart_team_data/mapped/spaceranger301/HEA_FOET14880396"
binned_output_path = f"{dir_path}/outs/binned_outputs/square_002um"
spaceranger_image_path = f"{dir_path}/outs/spatial"
source_image_path = "/rfs/project/rfs-iCNyzSAaucw/kk837/VisiumHD/source_images/C194_HEA_0_FFPE_1_STAN_1EAF2_s3_2024_04_23_13_40_00.tiff" 

In [3]:
adata = b2c.read_visium(binned_output_path, 
                        spaceranger_image_path=spaceranger_image_path,
                        source_image_path = source_image_path
                       )
adata.var_names_make_unique()
adata

anndata.py (1758): Variable names are not unique. To make them unique, call `.var_names_make_unique`.
anndata.py (1758): Variable names are not unique. To make them unique, call `.var_names_make_unique`.


AnnData object with n_obs × n_vars = 5032713 × 18085
    obs: 'in_tissue', 'array_row', 'array_col'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

# Pre-processing

In [4]:
# slightly filter the object
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_counts=1)
adata

AnnData object with n_obs × n_vars = 5001851 × 18051
    obs: 'in_tissue', 'array_row', 'array_col', 'n_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells'
    uns: 'spatial'
    obsm: 'spatial'

In [5]:
# mpp: microns per pixel
mpp = 0.3
b2c.scaled_he_image(adata, mpp=mpp, save_path=f"stardist/he.tiff")

Cropped spatial coordinates key: spatial_cropped_150_buffer
Image key: 0.3_mpp_150_buffer


In [6]:
adata.shape

(5001851, 18051)

# H&E segmentation - with different probability thresholds

In [7]:
prob_thresh_dict = {
    '0p01':0.01,
     '0p03':0.03,
    '0p05':0.05,
    '0p1':0.1,
    'default':None
}

for key,prob_thresh in prob_thresh_dict.items():
    print(f'### {key} ###')
    if key=='default':
        b2c.stardist(image_path=f"stardist/he.tiff",
                     labels_npz_path=f"stardist/he_{key}.npz", 
                     stardist_model="2D_versatile_he",
                    )
    else:
        b2c.stardist(image_path=f"stardist/he.tiff", 
                     labels_npz_path=f"stardist/he_{key}.npz", 
                     stardist_model="2D_versatile_he", 
                     prob_thresh=prob_thresh
                    )
    print('')

### 0p01 ###


2025-11-15 16:08:01.549609: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-15 16:08:01.551115: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 16:08:01.555530: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-15 16:08:01.566165: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763222881.580500 3418452 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763222881.58

Found model '2D_versatile_he' for 'StarDist2D'.


2025-11-15 16:09:03.184625: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
effective: block_size=(4096, 4096, 3), min_overlap=(128, 128, 0), context=(128, 128, 0)


...

Found 125657 objects

### 0p1 ###
Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
effective: block_size=(4096, 4096, 3), min_overlap=(128, 128, 0), context=(128, 128, 0)


100%|██████████| 49/49 [03:37<00:00,  4.44s/it]


Found 111372 objects

### default ###
Found model '2D_versatile_he' for 'StarDist2D'.
Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.692478, nms_thresh=0.3.
effective: block_size=(4096, 4096, 3), min_overlap=(128, 128, 0), context=(128, 128, 0)


100%|██████████| 49/49 [02:45<00:00,  3.38s/it]


Found 60505 objects

